In [15]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
from tqdm import tqdm
import numpy as np

In [123]:
response = requests.get("https://www.house.gov/representatives")
soup = BeautifulSoup(response.text, "html.parser")
table = soup.find_all("table", class_="table")
name_state_district_party_link = {}
df = pd.DataFrame(columns=["Name", "State", "District", "Party", "Link", "Issue Links"])

In [124]:
for letter in tqdm(table):
    state = letter.find_all("caption")[0].text.strip() #type: ignore
    for row in letter.find_all("tr")[1:]: #type: ignore
        district = row.find_all("td")[0].text.strip() #type: ignore
        name = row.find_all("td")[1].text.strip() #type: ignore
        party = row.find_all("td")[2].text.strip() #type: ignore
        raw_link = row.find_all("td")[1].find("a") #type: ignore
        link = str(raw_link)[9:str(raw_link).find('">')] #type: ignore
        if link.find("gov/") != -1:
            link = link[:-1]
        if len(link) > 0:
            name_state_district_party_link[link] = [name, state, district, party]
        df = df._append({"Name": name, "State": state, "District": district, "Party": party, "Link": link, "Issue Links": []}, ignore_index=True) #type: ignore

df = df.iloc[:441, :]

  0%|          | 0/81 [00:00<?, ?it/s]

100%|██████████| 81/81 [00:00<00:00, 149.03it/s]


In [137]:
idx = 0
for link in tqdm(df["Link"]):
    # print(link)
    link += "/issues"
    response = requests.get(link)
    soup = BeautifulSoup(response.text, "html.parser")
    issues_container = soup.find("div", class_="evo-views-row-container")
    issue_links = []
    if issues_container is None:
        try:
            alternate_container = soup.find("div", class_="issues-listed")
            for issue in alternate_container.find_all("a", class_="issues-item"): #type: ignore
                raw_issue = str(issue)[str(issue).find('href="') + 6:str(issue).find('">')]
                raw_issue = raw_issue[1:]
                issue = raw_issue[raw_issue.find("/"):]
                issue_links.append(issue)
        except AttributeError:
            pass
    else:
        for issue in issues_container.find_all("div", class_="h3 mt-0 font-weight-bold"): #type: ignore
            raw_issue = str(issue)[str(issue).find('<a href="') + 9:str(issue).find('</a>')]
            issue = raw_issue[1:raw_issue.find('">')]
            issue_links.append(issue[issue.find("/"):])
    df.at[idx, "Issue Links"] = [link + issue for issue in issue_links]
    # print(df.at[idx, "Name"], df.at[idx, "Issue Links"])
    idx += 1

100%|██████████| 441/441 [08:40<00:00,  1.18s/it]


In [138]:
df.head()

,Name,State,District,Party,Link,Issue Links
0,"Moore, Barry",Alabama,1st,R,https://barrymoore.house.gov,"[https://barrymoore.house.gov/issues/congress,..."
1,"Figures, Shomari",Alabama,2nd,D,https://figures.house.gov,"[https://figures.house.gov/issues/congress, ht..."
2,"Rogers, Mike",Alabama,3rd,R,https://mikerogers.house.gov,[https://mikerogers.house.gov/issues/Issue/?Is...
3,"Aderholt, Robert",Alabama,4th,R,https://aderholt.house.gov,"[https://aderholt.house.gov/issues/economy, ht..."
4,"Strong, Dale",Alabama,5th,R,https://strong.house.gov,"[https://strong.house.gov/issues/education, ht..."


In [139]:
issue_link_to_content = {}
full_issue_list = np.concatenate(df["Issue Links"].tolist()).tolist()
print(len(full_issue_list))

2912


In [140]:
# Barry Moore .find("div", class_="evo-issue__body") but placehold text
# Figures, .find("div", class_="evo-issue__body")
# Darren Soto, .find("div", class_="evo-issue__body") but has multiple <p> tags so that could be something to use to parse the content
# Mike Rogers, .find("div", class_="main-newscopy issue-copy").text.strip() as there is just placeholder text and no evo-issue__body div tag
for full_issue in tqdm(full_issue_list):
    #print(full_issue[-10:])
    response = requests.get(full_issue)
    soup = BeautifulSoup(response.text, "html.parser")
    content_container = soup.find("div", class_="evo-issue__body")
    if content_container is None:
        try:
            text = soup.find("div", class_="main-newscopy issue-copy").text.strip()
        except AttributeError:
            text = ""
    else:
        text = content_container.text.strip()
    if len(text) > 120:
        issue_link_to_content[full_issue] = text
    else:
        issue_link_to_content[full_issue] = ""
        #issue_link_to_content[full_issue] = content_container.text.strip()

issue_df = pd.DataFrame(list(issue_link_to_content.items()), columns=["Issue Link", "Content"])

100%|██████████| 2912/2912 [43:22<00:00,  1.12it/s] 


In [142]:
issue_df = issue_df[issue_df["Content"] != ""].reset_index(drop=True)
issue_df["Topic"] = issue_df["Issue Link"].apply(lambda x: x[x.find("/issues") + 8:].replace("-", " ").capitalize())
# issue_df = issue_df[["Topic", "Content", "Issue Link"]]
for link in tqdm(issue_df["Issue Link"]):
    values = name_state_district_party_link[link[:link.find("/issues")]]
    name, state, district, party = values[0], values[1], values[2], values[3]
    issue_df.loc[issue_df["Issue Link"] == link, "Name"] = name
    issue_df.loc[issue_df["Issue Link"] == link, "State"] = state
    issue_df.loc[issue_df["Issue Link"] == link, "District"] = district
    issue_df.loc[issue_df["Issue Link"] == link, "Party"] = party
issue_df = issue_df[["Name", "State", "District", "Party", "Content", "Issue Link"]]
issue_df

  0%|          | 0/1330 [00:00<?, ?it/s]

100%|██████████| 1330/1330 [00:03<00:00, 398.51it/s]


,Name,State,District,Party,Content,Issue Link
0,"Aderholt, Robert",Alabama,4th,R,I believe the majority of Alabamians support s...,https://aderholt.house.gov/issues/economy
1,"Aderholt, Robert",Alabama,4th,R,"""Knowledge is power."" Thomas Jefferson, Januar...",https://aderholt.house.gov/issues/education
2,"Aderholt, Robert",Alabama,4th,R,Energy becomes a more important topic each and...,https://aderholt.house.gov/issues/energy
3,"Aderholt, Robert",Alabama,4th,R,"""Congress shall make no law respecting an esta...",https://aderholt.house.gov/issues/faith-and-re...
4,"Aderholt, Robert",Alabama,4th,R,The Founding Fathers wisely included the 2nd A...,https://aderholt.house.gov/issues/gun-rights
...,...,...,...,...,...,...
1325,"Hageman, Harriet",Wyoming,At Large,R,"Wyoming is home to over 40,000 veterans and ad...",https://hageman.house.gov/issues/military-and-...
1326,"Hageman, Harriet",Wyoming,At Large,R,Wyoming’s farmers and ranchers feed the nation...,https://hageman.house.gov/issues/agriculture
1327,"Hageman, Harriet",Wyoming,At Large,R,"Border security is national security, and the ...",https://hageman.house.gov/issues/border-security
1328,"Hageman, Harriet",Wyoming,At Large,R,For over 30 years as a private attorney and no...,https://hageman.house.gov/issues/administrativ...


In [143]:
issue_df.to_csv("issue_df.csv", index=False)